In [17]:
import os
import pandas as pd
import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
def identify_significant_up_and_down_DEGs(DEG_df):
    '''Identify up- and down-regulated DEGs'''
    up_DEGs = DEG_df[(DEG_df['significant'] == True) & (DEG_df['log2FoldChange'] > 0)]['gene_id'].tolist()
    down_DEGs = DEG_df[(DEG_df['significant'] == True) & (DEG_df['log2FoldChange'] < 0)]['gene_id'].tolist()
    return up_DEGs, down_DEGs

In [10]:
def run_prerank_gsea(rank_df, gene_sets='MSigDB_Hallmark_2020', organism="Human", outdir=None):
    """
    Run GSEA prerank on a dataframe of genes ranked by log2FoldChange.
    
    Parameters
    ----------
    rank_df : pd.DataFrame
        DataFrame with columns ['gene_id', 'log2FoldChange'].
    gene_sets : str
        Name of gene set library (e.g., 'MSigDB_Hallmark_2020').
    organism : str
        'Human' or 'Mouse'
    outdir : str
        Directory to save results (optional)
    
    Returns
    -------
    gsea_res : pd.DataFrame
        GSEA results dataframe
    """
    # Prepare ranking: gene -> log2FC
    ranking = rank_df[['gene_id', 'log2FoldChange']].set_index('gene_id').squeeze()
    
    prerank_res = gp.prerank(
        rnk=ranking,
        gene_sets=gene_sets,
        processes=4,
        permutation_num=100,  # reduce for speed if needed
        outdir=outdir,
        format='png',
        seed=42,
        min_size=15,
        max_size=500,
        verbose=True
    )

    # Results
    gsea_res = prerank_res.res2d

    return gsea_res

In [13]:
results_dir = "gsea_prerank_results/"
os.makedirs(results_dir, exist_ok=True)

In [20]:
cell_types = ["Adipocyte", "Cardiomyocyte", "Endothelial",
        "Fibroblast", "LEC", "Lymphoid", "Myeloid", "Neuronal", "Pericyte"]
diseases = ["DCM", "HCM", "ICM"]
gene_sets = "MSigDB_Hallmark_2020"  # or "Hallmark_2023"

In [21]:
for cell_type in cell_types:

    for disease in diseases:

        # Load DEG
        df = pd.read_csv(f"pydeseq2_results/{cell_type}_disease_{disease}_vs_ND_results.csv", index_col=0)

        # Create prerank dataframe
        # Use log2FoldChange as the ranking metric
        prerank_df = df[['gene_id', 'log2FoldChange']].dropna()
        prerank_df = prerank_df.sort_values('log2FoldChange', ascending=False)

        # Save as .rnk file (optional)
        rnk_file = f"{results_dir}/{cell_type}_{disease}_prerank.rnk"
        prerank_df.to_csv(rnk_file, sep='\t', header=False, index=False)

        # Run preranked GSEA
        pre_res = gp.prerank(
            rnk=rnk_file,
            gene_sets=gene_sets,
            processes=4,
            permutation_num=100,  # reduce for speed; increase for publication-quality
            outdir=None,          # do not write files automatically
            seed=42,
            verbose=True
        )

        gsea_res = pre_res.res2d
        gsea_res['cell_type'] = cell_type
        gsea_res['disease'] = disease

        # Save results
        gsea_res.to_csv(f"{results_dir}/{cell_type}_{disease}_prerank_GSEA.csv")

        print(f"Finished preranked GSEA for {cell_type} - {disease}")

/mnt/data1/william/tmp/ipykernel_1872604/996878316.py:18: DeprecationWarning: processes is deprecated; use threads
  pre_res = gp.prerank(
2025-09-25 18:07:33,599 [INFO] Parsing data files for GSEA.............................
2025-09-25 18:07:33,600 [INFO] Enrichr library gene sets already downloaded in: /home/william/.cache/gseapy, use local file
2025-09-25 18:07:33,610 [INFO] 0001 gene_sets have been filtered out when max_size=500 and min_size=15
2025-09-25 18:07:33,611 [INFO] 0049 gene_sets used for further statistical testing.....
2025-09-25 18:07:33,611 [INFO] Start to run GSEA...Might take a while..................
2025-09-25 18:07:33,935 [INFO] Congratulations. GSEApy runs successfully................

/mnt/data1/william/tmp/ipykernel_1872604/996878316.py:18: DeprecationWarning: processes is deprecated; use threads
  pre_res = gp.prerank(
2025-09-25 18:07:34,061 [INFO] Parsing data files for GSEA.............................
2025-09-25 18:07:34,062 [INFO] Enrichr library gene s

Finished preranked GSEA for Adipocyte - DCM


2025-09-25 18:07:34,444 [INFO] Congratulations. GSEApy runs successfully................

/mnt/data1/william/tmp/ipykernel_1872604/996878316.py:18: DeprecationWarning: processes is deprecated; use threads
  pre_res = gp.prerank(
2025-09-25 18:07:34,598 [WARNING] Duplicated values found in preranked stats: 0.26% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2025-09-25 18:07:34,598 [INFO] Parsing data files for GSEA.............................
2025-09-25 18:07:34,599 [INFO] Enrichr library gene sets already downloaded in: /home/william/.cache/gseapy, use local file
2025-09-25 18:07:34,604 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2025-09-25 18:07:34,604 [INFO] 0050 gene_sets used for further statistical testing.....
2025-09-25 18:07:34,605 [INFO] Start to run GSEA...Might take a while..................


Finished preranked GSEA for Adipocyte - HCM


2025-09-25 18:07:35,055 [INFO] Congratulations. GSEApy runs successfully................

/mnt/data1/william/tmp/ipykernel_1872604/996878316.py:18: DeprecationWarning: processes is deprecated; use threads
  pre_res = gp.prerank(
2025-09-25 18:07:35,210 [INFO] Parsing data files for GSEA.............................
2025-09-25 18:07:35,211 [INFO] Enrichr library gene sets already downloaded in: /home/william/.cache/gseapy, use local file
2025-09-25 18:07:35,216 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2025-09-25 18:07:35,217 [INFO] 0050 gene_sets used for further statistical testing.....
2025-09-25 18:07:35,218 [INFO] Start to run GSEA...Might take a while..................


...

Finished preranked GSEA for Pericyte - DCM


2025-09-25 18:07:47,574 [INFO] Congratulations. GSEApy runs successfully................

/mnt/data1/william/tmp/ipykernel_1872604/996878316.py:18: DeprecationWarning: processes is deprecated; use threads
  pre_res = gp.prerank(
2025-09-25 18:07:47,694 [INFO] Parsing data files for GSEA.............................
2025-09-25 18:07:47,695 [INFO] Enrichr library gene sets already downloaded in: /home/william/.cache/gseapy, use local file
2025-09-25 18:07:47,701 [INFO] 0001 gene_sets have been filtered out when max_size=500 and min_size=15
2025-09-25 18:07:47,702 [INFO] 0049 gene_sets used for further statistical testing.....
2025-09-25 18:07:47,702 [INFO] Start to run GSEA...Might take a while..................


Finished preranked GSEA for Pericyte - HCM


2025-09-25 18:07:48,022 [INFO] Congratulations. GSEApy runs successfully................



Finished preranked GSEA for Pericyte - ICM
